In [1]:
# Task 10 : Simple WGAN-GP


import torch
import torch.nn as nn
import torch.optim as optim


class Generator(nn.Module):

    def __init__(self):
        super(Generator, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(100,128),
            nn.ReLU(),

            nn.Linear(128,256),
            nn.ReLU(),

            nn.Linear(256,784),
            nn.Tanh()
        )

    def forward(self,x):
        return self.model(x)



class Critic(nn.Module):

    def __init__(self):
        super(Critic,self).__init__()

        self.model = nn.Sequential(
            nn.Linear(784,256),
            nn.LeakyReLU(0.2),

            nn.Linear(256,128),
            nn.LeakyReLU(0.2),

            nn.Linear(128,1)
        )

    def forward(self,x):
        return self.model(x)



def gradient_penalty(critic,real,fake):

    batch_size = real.size(0)

    alpha = torch.rand(batch_size,1)

    alpha = alpha.expand_as(real)

    interpolated = alpha*real + (1-alpha)*fake

    interpolated.requires_grad_(True)

    score = critic(interpolated)

    gradients = torch.autograd.grad(
        outputs=score,
        inputs=interpolated,
        grad_outputs=torch.ones_like(score),
        create_graph=True,
        retain_graph=True
    )[0]

    gradients = gradients.view(batch_size,-1)

    gp = ((gradients.norm(2,dim=1)-1)**2).mean()

    return gp



generator = Generator()

critic = Critic()



g_optimizer = optim.Adam(generator.parameters(),lr=0.0001)

c_optimizer = optim.Adam(critic.parameters(),lr=0.0001)



batch_size = 16

real_images = torch.randn(batch_size,784)

noise = torch.randn(batch_size,100)

fake_images = generator(noise)



real_score = critic(real_images)

fake_score = critic(fake_images.detach())

gp = gradient_penalty(critic,real_images,fake_images.detach())

critic_loss = -(torch.mean(real_score)
                - torch.mean(fake_score)) + 10*gp


generator_loss = -torch.mean(critic(fake_images))



print("Critic Loss :",critic_loss.item())

print("Generator Loss :",generator_loss.item())

ModuleNotFoundError: No module named 'torch'